## Tool Calling Agent
Here we can use the Python SDK to develop the simple tool calling agent, then save the agent to a config.yaml and run it from there.

In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [2]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

In [ ]:
from nat.llm.nim_llm import NIMModelConfig
from nat.plugins.langchain.tools.code_generation_tool import CodeGenerationTool
from nat.plugins.langchain.tools.code_generation_tool import code_generation_tool
from nat.plugins.langchain.tools.wikipedia_search import WikiSearchToolConfig
from nat.plugins.langchain.tools.wikipedia_search import wiki_search
from nat.tool.datetime_tools import CurrentTimeToolConfig
from nat.tool.datetime_tools import current_datetime
from nat.utils.sdk.nat_agent import NatToolCallingAgent
from nat.utils.sdk.nat_llm import NatLLM
from nat.utils.sdk.nat_tool import NatTool

llm = NatLLM(
    config=NIMModelConfig(
        model_name="nvdev/meta/llama-3.1-70b-instruct",
        temperature=0,
        max_tokens=250,
    ),
    name="nim_llm",
)

# Define a list of tools
wikipedia_search_tool = NatTool(
    config=WikiSearchToolConfig(max_results=3),
    function=wiki_search,
    name="wiki_search",
)
current_time_tool = NatTool(
    config=CurrentTimeToolConfig(),
    function=current_datetime,
    name="current_datetime",
)
generate_code_tool = NatTool(
    config=CodeGenerationTool(
        programming_language="Python",
        description="Useful to generate Python code. For any questions about code generation, you must only use this tool!",  # noqa: E501
        llm_name=llm.llm_name,
        verbose=True,
    ),
    function=code_generation_tool,
    name="code_generation",
)

agent = NatToolCallingAgent(
    tools=[wikipedia_search_tool, current_time_tool, generate_code_tool],
    llm=llm,
    verbose=True,
    handle_tool_errors=True,
)

/Users/spastoriza/Documents/Programming/public/nat-official/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
await agent.prompt("Who was Djikstra?")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


'Edsger Dijkstra was a Dutch computer scientist who developed the paradigm of structured programming for writing computer programs and made significant contributions to the field of computer science, including the development of the programming language ALGOL 60 and the concept of concurrent programming.'

In [5]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
agent.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  internet_search:
    _type: tavily_internet_search
  haystack_chitchat_agent:
    _type: haystack_chitchat_agent
    llm_name: meta/llama-3.1-405b-instruct

function_groups:
  calculator:
    _type: calculator

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.3-70b-instruct
    max_tokens: 4096
    temperature: 0.0

workflow:
  _type: rewoo_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - internet_search
  - haystack_chitchat_agent
  - calculator
  tool_call_max_retries: 3



In [ ]:
from pathlib import Path

from nat.data_models.dataset_handler import EvalDatasetJsonConfig
from nat.eval.rag_evaluator.register import RagasEvaluatorConfig
from nat.eval.rag_evaluator.register import register_ragas_evaluator
from nat.utils.sdk.nat_evaluator import NatEvaluator
from nat.utils.sdk.nat_general_evaluator import NatGeneralEvaluator
from nat.utils.sdk.nat_targeted_evaluator import NatTargetedEvaluator

path_to_dataset = Path(os.path.curdir, "../../../../", "examples/agents/data/wikipedia.json").resolve()

evaluator = NatGeneralEvaluator(
    output_dir=Path(".tmp/nat/examples/tool_calling_agent/"),
    dataset=EvalDatasetJsonConfig(file_path=path_to_dataset),
)
accuracy_evaluator = NatTargetedEvaluator(config=RagasEvaluatorConfig(llm_name=llm.llm_name, metric="AnswerAccuracy"),
                                          name="accuracy",
                                          evaluator=register_ragas_evaluator)

relevance_evaluator = NatTargetedEvaluator(config=RagasEvaluatorConfig(llm_name=llm.llm_name,
                                                                       metric="ContextRelevance"),
                                           name="relevance",
                                           evaluator=register_ragas_evaluator)

response_groundedness_evaluator = NatTargetedEvaluator(config=RagasEvaluatorConfig(llm_name=llm.llm_name,
                                                                                   metric="ResponseGroundedness"),
                                                       name="groundedness",
                                                       evaluator=register_ragas_evaluator)

evaluator = NatEvaluator(general_evaluator=evaluator,
                         evaluators=[accuracy_evaluator, relevance_evaluator, response_groundedness_evaluator])

agent.add_evaluator(evaluator)

In [7]:
path_to_yaml = Path(os.getcwd(), "config", "eval_config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
agent.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

functions:
  internet_search:
    _type: tavily_internet_search
  haystack_chitchat_agent:
    _type: haystack_chitchat_agent
    llm_name: meta/llama-3.1-405b-instruct

function_groups:
  calculator:
    _type: calculator

llms:
  nim_llm:
    _type: nim
    model: nvdev/meta/llama-3.3-70b-instruct
    max_tokens: 4096
    temperature: 0.0

workflow:
  _type: rewoo_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - internet_search
  - haystack_chitchat_agent
  - calculator
  tool_call_max_retries: 3

eval:
  general:
    output_dir: .tmp/nat/examples/rewoo_agent
    dataset:
      _type: json
      file_path: /Users/spastoriza/Documents/Programming/public/nat-official/examples/agents/data/rewoo.json
  evaluators:
    accuracy:
      _type: ragas
      llm_name: nim_llm
      metric: AnswerAccuracy
    relevance:
      _type: ragas
      llm_name: nim_llm
      metric: ContextRelevance
    groundedness:
      _type: ragas
      llm_name: nim_llm
      metric: ResponseGroundedn

In [8]:
await agent.evaluate()

Evaluating Ragas nv_accuracy:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating Ragas nv_accuracy: 100%|██████████| 5/5 [00:03<00:00,  1.42it/s]
